## extract ca coordinates

In [1]:
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pandas as pd

DATA_DIR = Path("/home/jupyter/gcs_mount/data/ca_coords")

files = sorted(list(DATA_DIR.glob("*.npy")))
print("Total files:", len(files))

sample = np.load(files[0])
print("Sample file:", files[0].name)
print("Shape:", sample.shape)

Total files: 246
Sample file: 5-hydroxytryptamine_receptor_1B_4IAQ_A_rep1_85.npy
Shape: (2500, 302, 3)


In [2]:
# === determine optimal fixed number of ca atoms to extract ===
ca_counts = []

for f in files:
    arr = np.load(f)
    ca_counts.append(arr.shape[1])

ca_counts = np.array(ca_counts)

print("Total systems:", len(ca_counts))
print("Min:", ca_counts.min())
print("Max:", ca_counts.max())

# count how many below thresholds
thresholds = [100, 200, 250, 280, 290, 300]

for t in thresholds:
    print(f"Below {t}: {np.sum(ca_counts < t)}")

print(f"Sorted counts (first 20): {sorted(ca_counts)[:20]}")

Total systems: 246
Min: 72
Max: 1030
Below 100: 1
Below 200: 1
Below 250: 1
Below 280: 11
Below 290: 30
Below 300: 82
Sorted counts (first 20): [np.int64(72), np.int64(255), np.int64(259), np.int64(259), np.int64(259), np.int64(259), np.int64(275), np.int64(275), np.int64(275), np.int64(275), np.int64(275), np.int64(288), np.int64(288), np.int64(289), np.int64(289), np.int64(289), np.int64(289), np.int64(289), np.int64(289), np.int64(289)]


In [3]:
# === helpers for alignment and sampling ===
def kabsch_align_centered(mobile, reference):
    mobile_centered = mobile - mobile.mean(axis=0)
    reference_centered = reference - reference.mean(axis=0)

    covariance = mobile_centered.T @ reference_centered
    V, _, Wt = np.linalg.svd(covariance)

    if np.linalg.det(V @ Wt) < 0:
        V[:, -1] *= -1

    rotation = V @ Wt
    mobile_aligned_centered = mobile_centered @ rotation
    return mobile_aligned_centered

def align_trajectory(coords):
    """
    coords: (T, N, 3)
    returns: aligned (T, N, 3)
    """
    ref = coords[0]
    aligned = []

    for frame in coords:
        aligned_frame = kabsch_align_centered(frame, ref)
        aligned.append(aligned_frame)

    return np.stack(aligned, axis=0)

def uniform_subsample_ca(coords, target_n):
    frames, n_ca, _ = coords.shape
    
    if n_ca == target_n:
        return coords
    
    indices = np.linspace(0, n_ca - 1, target_n)
    indices = np.round(indices).astype(int)
    
    return coords[:, indices, :]

In [4]:
# === set N ===
N_CA = 288
DATA_DIR = Path("/home/jupyter/gcs_mount/data/ca_coords")
OUT_DIR = Path("/home/jupyter/gcs_mount/data/ca_coords_fixed_288")
OUT_DIR.mkdir(exist_ok=True)

files = sorted(DATA_DIR.glob("*.npy"))

for f in tqdm(files):
    coords = np.load(f)  # (T, N, 3)

    if coords.shape[1] < N_CA:
        continue  # skip small systems

    # align
    coords_aligned = align_trajectory(coords)

    # subsample
    coords_fixed = uniform_subsample_ca(coords_aligned, N_CA)

    # flatten
    coords_flat = coords_fixed.reshape(coords_fixed.shape[0], -1)

    # save
    # np.save(OUT_DIR / f.name, coords_flat)

100%|██████████| 246/246 [01:06<00:00,  3.68it/s]


## build new csv from existing csv

In [5]:
# === load processed CA files ===
CA_DIR = Path("/home/jupyter/gcs_mount/data/ca_coords_fixed_288")
ca_files = sorted([f.name for f in CA_DIR.glob("*.npy")])

print("Total CA-fixed files:", len(ca_files))
print("Example files:", ca_files[:5])

# load full trajectory-level dataset
base_df = pd.read_csv("/home/jupyter/cs229-md-prediction/data/processed/data_processed_v3_base_dataset_deduped.csv")

print("Total trajectories in base dataset:", len(base_df))
base_df.head()

Total CA-fixed files: 235
Example files: ['5-hydroxytryptamine_receptor_1B_4IAQ_A_rep1_85.npy', '5-hydroxytryptamine_receptor_1B_4IAR_A_rep1_87.npy', '5-hydroxytryptamine_receptor_1B_4IAR_A_rep1_90.npy', '5-hydroxytryptamine_receptor_2B_4IB4_A_rep1_92.npy', '5-hydroxytryptamine_receptor_2B_4IB4_A_rep1_94.npy']
Total trajectories in base dataset: 247


,receptor,rep,n_frames_total,n_frames_used,early_frac,n_windows,rmsd_mean,rmsd_std,rmsd_max,rg_mean,...,tm3_tm6_std,simID,traj_file,top_file,label,dominant_cluster_frac,silhouette,y,early_ts_path,early_frames_path
0,5-hydroxytryptamine_receptor_1B~4IAQ_A,1,2500,1200,0.5,1,4.474691,1.174720,6.749105,22.182590,...,NaN,85,5-hydroxytryptamine_receptor_1B~4IAQ_A~unknown...,5-hydroxytryptamine_receptor_1B~4IAQ_A~85~1082...,multi_state,0.574583,0.500110,1.0,early_ts_5-hydroxytryptamine_receptor_1B_4IAQ_...,early_frames_5-hydroxytryptamine_receptor_1B_4...
1,5-hydroxytryptamine_receptor_1B~4IAR_A,1,2500,1200,0.5,1,2.547754,0.408294,4.311748,21.854927,...,NaN,87,5-hydroxytryptamine_receptor_1B~4IAR_A~unknown...,5-hydroxytryptamine_receptor_1B~4IAR_A~87~1084...,single_state,0.639167,0.426066,0.0,early_ts_5-hydroxytryptamine_receptor_1B_4IAR_...,early_frames_5-hydroxytryptamine_receptor_1B_4...
2,5-hydroxytryptamine_receptor_1B~4IAR_A,1,2500,1200,0.5,1,2.601540,0.409481,4.350486,22.018714,...,NaN,90,5-hydroxytryptamine_receptor_1B~4IAR_A~unknown...,5-hydroxytryptamine_receptor_1B~4IAR_A~90~1087...,single_state,0.789167,0.546720,0.0,early_ts_5-hydroxytryptamine_receptor_1B_4IAR_...,early_frames_5-hydroxytryptamine_receptor_1B_4...
3,5-hydroxytryptamine_receptor_2B~4IB4_A,1,2500,1200,0.5,1,2.911906,0.504053,4.357796,22.523872,...,2.253397,92,5-hydroxytryptamine_receptor_2B~4IB4_A~unknown...,5-hydroxytryptamine_receptor_2B~4IB4_A~unknown...,single_state,0.663750,0.417202,0.0,early_ts_5-hydroxytryptamine_receptor_2B_4IB4_...,early_frames_5-hydroxytryptamine_receptor_2B_4...
4,5-hydroxytryptamine_receptor_2B~4IB4_A,1,2500,1200,0.5,1,3.474734,0.516913,4.636500,22.303306,...,1.804635,94,5-hydroxytryptamine_receptor_2B~4IB4_A~unknown...,5-hydroxytryptamine_receptor_2B~4IB4_A~unknown...,multi_state,0.600417,0.539242,1.0,early_ts_5-hydroxytryptamine_receptor_2B_4IB4_...,early_frames_5-hydroxytryptamine_receptor_2B_4...


In [6]:
# === construct CA filename from trajectory metadata ===
base_df["chunk_file"] = (
    base_df["receptor"].str.replace("~", "_")
    + "_rep"
    + base_df["rep"].astype(str)
    + "_"
    + base_df["simID"].astype(str)
    + ".npy"
)

# check how many match CA files
matches = base_df["chunk_file"].isin(ca_files).sum()

print("Matching trajectories:", matches)
print("CA files total:", len(ca_files))

Matching trajectories: 235
CA files total: 235


In [7]:
# === build train/test receptor sets ===
train_receptors = pd.read_csv("/home/jupyter/cs229-md-prediction/data/processed/train_receptors_v3_deduped.csv")
test_receptors = pd.read_csv("/home/jupyter/cs229-md-prediction/data/processed/test_receptors_v3_deduped.csv")

# remove duplicates just to be safe
train_receptors = train_receptors.drop_duplicates(subset="receptor")
test_receptors  = test_receptors.drop_duplicates(subset="receptor")

# build split lookup
split_lookup = {}

for r in train_receptors["receptor"]:
    split_lookup[r] = "train"

for r in test_receptors["receptor"]:
    split_lookup[r] = "test"

# build family lookup
family_lookup = {}

for _, row in train_receptors.iterrows():
    family_lookup[row["receptor"]] = row["family"]

for _, row in test_receptors.iterrows():
    family_lookup[row["receptor"]] = row["family"]

print("Train receptors:", len(train_receptors))
print("Test receptors:", len(test_receptors))
print("Overlap:",
      set(train_receptors["receptor"]).intersection(
          set(test_receptors["receptor"])
      ))

base_df["split"] = base_df["receptor"].map(split_lookup)
base_df["family"] = base_df["receptor"].map(family_lookup)

Train receptors: 105
Test receptors: 39
Overlap: set()


In [9]:
# === assign split directly ===
train_set = set(train_receptors["receptor"])
test_set = set(test_receptors["receptor"])

def assign_split(receptor):
    if receptor in train_set:
        return "train"
    elif receptor in test_set:
        return "test"
    else:
        return None

base_df["split"] = base_df["receptor"].apply(assign_split)

print(base_df["split"].value_counts())

split
train    190
test      57
Name: count, dtype: int64


In [10]:
# === filter to CA-fixed trajectories ===
final_df = base_df[base_df["chunk_file"].isin(ca_files)].copy()
final_df["y"] = (final_df["label"] == "multi_state").astype(int)
train_final = final_df[final_df["split"] == "train"]
test_final = final_df[final_df["split"] == "test"]

print("Final trajectories:", len(final_df))
print("CA files:", len(ca_files))
print("Multi fraction:", final_df["y"].mean())
print("Train size:", len(train_final))
print("Test size:", len(test_final))

Final trajectories: 235
CA files: 235
Multi fraction: 0.2978723404255319
Train size: 180
Test size: 55


## save train/test split csvs

In [11]:
train_final = final_df[final_df["split"] == "train"].copy()
test_final  = final_df[final_df["split"] == "test"].copy()

print("Train size:", len(train_final))
print("Test size:", len(test_final))

Train size: 180
Test size: 55


In [ ]:
# train_final[["chunk_file", "y"]].to_csv(
#     OUT_DIR / "train_ca_fixed288.csv",
#     index=False
# )

# test_final[["chunk_file", "y"]].to_csv(
#     OUT_DIR / "test_ca_fixed288.csv",
#     index=False
# )

# print("Saved:")
# print(" - train_ca_fixed288.csv")
# print(" - test_ca_fixed288.csv")

# REMOVING CONFLICTING LABELS FOR NEW SINGLE_LABEL DATASET

In [21]:
# === build single_label dataset (prefer multi_state) ===

SINGLE_LABEL_DIR = Path("/home/jupyter/gcs_mount/data/ca_coords_fixed_288_single_label")
SINGLE_LABEL_DIR.mkdir(exist_ok=True)

selected_rows = []

for receptor, group in final_df.groupby("receptor"):

    # prefer multi_state if it exists
    if "multi_state" in group["label"].values:
        selected_rows.append(group[group["label"] == "multi_state"])
    else:
        selected_rows.append(group)

single_label_df = pd.concat(selected_rows).copy()

print("Original trajectories:", len(final_df))
print("Remaining trajectories:", len(single_label_df))
print("Unique receptors:", single_label_df["receptor"].nunique())
print(single_label_df["label"].value_counts())

Original trajectories: 235
Remaining trajectories: 186
Unique receptors: 136
label
single_state    116
multi_state      70
Name: count, dtype: int64


In [18]:
# === copy corresponding .npy files ===

SRC_DIR = Path("/home/jupyter/gcs_mount/data/ca_coords_fixed_288")

for f in tqdm(single_label_df["chunk_file"]):
    shutil.copy(SRC_DIR / f, SINGLE_LABEL_DIR / f)

100%|██████████| 186/186 [00:56<00:00,  3.28it/s]


In [24]:
# === build train/test receptor sets ===
train_receptors = pd.read_csv("/home/jupyter/cs229-md-prediction/data/metadata/splits/protein_level_train.csv")
test_receptors = pd.read_csv("/home/jupyter/cs229-md-prediction/data/metadata/splits/protein_level_test.csv")

# remove duplicates just to be safe
train_receptors = train_receptors.drop_duplicates(subset="receptor")
test_receptors  = test_receptors.drop_duplicates(subset="receptor")

# build split lookup
split_lookup = {}

for r in train_receptors["receptor"]:
    split_lookup[r] = "train"

for r in test_receptors["receptor"]:
    split_lookup[r] = "test"

# build family lookup
family_lookup = {}

for _, row in train_receptors.iterrows():
    family_lookup[row["receptor"]] = row["family"]

for _, row in test_receptors.iterrows():
    family_lookup[row["receptor"]] = row["family"]

print("Train receptors:", len(train_receptors))
print("Test receptors:", len(test_receptors))
print("Overlap:",
      set(train_receptors["receptor"]).intersection(
          set(test_receptors["receptor"])
      ))

single_label_df["split"] = single_label_df["receptor"].map(split_lookup)
single_label_df["family"] = single_label_df["receptor"].map(family_lookup)

# === build train/test splits ===
single_label_df["y"] = (single_label_df["label"] == "multi_state").astype(int)

train_single = single_label_df[single_label_df["split"] == "train"].copy()
test_single  = single_label_df[single_label_df["split"] == "test"].copy()

print("Train size:", len(train_single))
print("Test size:", len(test_single))
print("Multi fraction:", single_label_df["y"].mean())

Train receptors: 115
Test receptors: 28
Overlap: set()
Train size: 150
Test size: 36
Multi fraction: 0.3763440860215054


In [25]:
# === save metadata ===

# train_single[["chunk_file","y"]].to_csv(
#     SINGLE_LABEL_DIR / "train_ca_fixed288_single_label.csv",
#     index=False
# )

# test_single[["chunk_file","y"]].to_csv(
#     SINGLE_LABEL_DIR / "test_ca_fixed288_single_label.csv",
#     index=False
# )

train_single[["chunk_file","y"]].to_csv(
    SINGLE_LABEL_DIR / "train_ca_fixed288_single_label_protein_level.csv",
    index=False
)

test_single[["chunk_file","y"]].to_csv(
    SINGLE_LABEL_DIR / "test_ca_fixed288_single_label_protein_level.csv",
    index=False
)

print("Saved:")
print(" - train_ca_fixed288_single_label_protein_level.csv")
print(" - test_ca_fixed288_single_label_protein_level.csv")

Saved:
 - train_ca_fixed288_single_label_protein_level.csv
 - test_ca_fixed288_single_label_protein_level.csv
